# Audio-conditioned album-cover diffusion (Colab)

Trains a LoRA adapter on Stable Diffusion 1.5 conditioned on CLAP audio embeddings.

Data layout expected on Drive:
```
cs159project/
  data/artist1/audio/*.mp3   data/artist1/covers/*.jpg
  data/artist2/audio/*.mp3   data/artist2/covers/*.jpg
  data/artist3/audio/*.mp3   data/artist3/covers/*.jpg
```
Audio stem and cover stem must match (e.g. `song01.mp3` ↔ `song01.jpg`).

## 1. GPU check + mount Drive

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/cs159project

## 2. Install deps

In [ ]:
!pip install -q -r requirements.txt

## 3. Extract CLAP embeddings + VAE latents (one-time)

In [ ]:
!python prepare_data.py --data-root data --cache-dir cache

## 4. Train

With 60 pairs and batch size 4 you get 15 steps/epoch. 200 epochs ≈ 3000 steps — usually enough to see audio→style coupling. Bump `--epochs` if loss is still dropping.

In [ ]:
!python train.py \
  --cache-dir cache \
  --out-dir checkpoints \
  --epochs 200 \
  --batch-size 4 \
  --lr 1e-4 \
  --lora-rank 8 \
  --num-tokens 8 \
  --cfg-drop 0.1 \
  --save-every 50 \
  --mixed-precision fp16

## 5. Generate a cover from an mp3

In [ ]:
import glob, os
ckpt = sorted(glob.glob('checkpoints/ckpt_e*'))[-1]
test_mp3 = sorted(glob.glob('data/artist1/audio/*.mp3'))[0]
print('ckpt:', ckpt)
print('mp3 :', test_mp3)
!python generate.py --ckpt {ckpt} --audio "{test_mp3}" --out outputs/sample.png --steps 30 --guidance 5.0 --size 256

In [ ]:
from PIL import Image
Image.open('outputs/sample.png')